In [1]:
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
llm=ChatOpenAI()

In [2]:
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
model=HuggingFaceEndpoint(
    model="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)
llm=ChatHuggingFace(llm=model)

In [3]:
import requests
from langchain_core.tools import tool

In [4]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool=DuckDuckGoSearchRun()

In [5]:
@tool
def get_weather_data(city:str)->str:
    """
    This function fetches the current weather data for a given city.
    """
    url=f"https://api.weatherstack.com/current?access_key=4d1d8e207a8c845a52df8a67bf3623e&query={city}"
    response=requests.get(url)
    return response.json()

In [6]:
from langchain_classic.agents import AgentExecutor,create_react_agent
from langchain_classic import hub

c:\Users\Admin\OneDrive\Desktop\New folder\LANGCHAIN_MODELS\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Pull the ReAct prompt from hub
# import os
# os.environ["HUGGINGFACEHUB_API_TOKEN"]=""
import os
from dotenv import load_dotenv
load_dotenv()
token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
prompt=hub.pull("hwchase17/react",dangerously_pull_public_prompt=True)

In [8]:
from langchain_core.prompts import PromptTemplate

react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(react_template)

In [9]:
# Create ReAct agent manually with the pulled prompt
agent=create_react_agent(
    llm=llm,
    tools=[search_tool,get_weather_data],
    prompt=prompt
)

In [13]:
# Wrap it with AgentExecutor
agent_executor=AgentExecutor(
    agent=agent,
    tools=[search_tool,get_weather_data],
    verbose=True,
    handle_parsing_errors=True   
)

In [14]:
# invoke
response=agent_executor.invoke({"input":"Find capital of MP andfind its weather conditions"})
response



> Entering new AgentExecutor chain...
Thought: To find the capital of Madhya Pradesh and its current weather conditions, I need to perform two separate actions.

Action: get_weather_data
Action Input: city="Bhopal" (as Bhopal is the capital of Madhya Pradesh){'success': False, 'error': {'code': 101, 'type': 'invalid_access_key', 'info': 'You have not supplied a valid API Access Key. [Technical Support: support@apilayer.com]'}}It seems I've encountered an error while trying to fetch the weather data for Bhopal. This could be due to an invalid or missing API access key. I'll need to take an alternative approach to find the capital of Madhya Pradesh and its weather conditions.

Action: duckduckgo_search
Action Input: capital of Madhya Pradesh and its weather conditionsIndore (/ ɪnˈdɔːr / ⓘ; ISO: Indaura, Hindi: [ɪn̪d̪ɔːr]) is the largest city of the Indian state of Madhya Pradesh where it serves as the capital and the administrative headquarters of the eponymous district and division. M

DDGSException: DNSError: DNSError('error sending request for url (https://wt.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=capital%20of%20Madhya%20Pradesh%20and%20its%20weather%20conditions) > error resolving DNS > DNS error: no records found for Query { name: Name("wt.wikipedia.org."), query_type: A, query_class: IN }')

In [ ]:
print(response)

In [ ]:
response["output"]